# Test

In [1]:
import torch
import torch.nn as nn
from PIL import Image
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor, AutoModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---------0) LOAD MODELS (FROZEN) ---------
LLM_NAME = "Qwen/Qwen2.5-0.5B"
VISION_NAME = "google/siglip-base-patch16-224"

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME, dtype=torch.float16
).to(DEVICE)
llm.requires_grad_(False)
llm.eval()

vision_processor = AutoProcessor.from_pretrained(VISION_NAME, use_fast=True)
vision_model = AutoModel.from_pretrained(
    VISION_NAME, dtype=torch.float16
).to(DEVICE)
vision_model.requires_grad_(False)
vision_model.eval()

# ---------1) ADD IMAGE TOKEN ---------
IMAGE_TOKEN = "<image>"
if IMAGE_TOKEN not in tokenizer.get_vocab():
    tokenizer.add_special_tokens({"additional_special_tokens": [IMAGE_TOKEN]})
    llm.resize_token_embeddings(len(tokenizer))

# ---------2) CONNECTOR (SHALLOW MLP) ---------
# SigLIP output dimension
vision_dim = vision_model.config.vision_config.hidden_size
llm_dim = llm.config.hidden_size

connector = nn.Sequential(
    nn.Linear(vision_dim, 4096),
    nn.GELU(),
    nn.Linear(4096, llm_dim)
).to(DEVICE).half()

optimizer = torch.optim.AdamW(connector.parameters(), lr=1e-3)

# ---------3) IMAGE → VISUAL TOKENS ---------
# @torch.no_grad()
def encode_image(img, K=32):
    with torch.no_grad():
        inputs = vision_processor(images=img, return_tensors="pt").to(DEVICE)
        feats = vision_model.vision_model(pixel_values=inputs["pixel_values"]).last_hidden_state  # shape: [1, seq_len, vision_dim]
        feats = feats[:, :K, :]  # truncate to K tokens
    return connector(feats)   # project to LLM dim

# ---------4) BUILD MULTIMODAL EMBEDDINGS ---------
def build_inputs(img, text, K):
    vis_embeds = encode_image(img, K)  # [1, K, llm_dim]
    text_input = f"{IMAGE_TOKEN} {text}"
    ids = tokenizer(text_input, return_tensors="pt").input_ids.to(DEVICE)
    text_embeds = llm.get_input_embeddings()(ids)

    # replace IMAGE_TOKEN with visual embeddings
    idx = (ids == tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)).nonzero()[0, 1]
    embeds = torch.cat([text_embeds[:, :idx], vis_embeds, text_embeds[:, idx + 1:]], dim=1)
    attention_mask = torch.ones(embeds.size()[:-1], device=DEVICE)
    return embeds, attention_mask, idx

# ---------5) TRAIN STEP (ALIGNMENT) ---------
img = Image.open("example.jpg").convert("RGB")
caption = "A dog running in a grassy field."
K = 32

labels = tokenizer(caption, return_tensors="pt").input_ids.to(DEVICE)
inputs_embeds, attention_mask, idx = build_inputs(img, caption, K)

text_input = f"{IMAGE_TOKEN} {caption}"
ids = tokenizer(text_input, return_tensors="pt").input_ids.to(DEVICE)
idx = (ids == tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)).nonzero()[0, 1]

labels = torch.cat([
    labels[:, :idx],  # text before image
    torch.full((1, K), -100, device=DEVICE, dtype=labels.dtype),  # ignore image tokens
    labels[:, idx:]  # text after image
], dim=1)

out = llm(inputs_embeds=inputs_embeds, attention_mask=attention_mask, labels=labels)
loss = out.loss

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Loss:", loss.item())

# ---------6) INFERENCE TEST ---------
with torch.no_grad():
    prompt = "What is in this image?"
    embeds, attention_mask, idx = build_inputs(img, prompt, K=K)
    generated = llm.generate(inputs_embeds=embeds, attention_mask = attention_mask, max_new_tokens=40, pad_token_id=tokenizer.pad_token_id)

print(tokenizer.decode(generated[0], skip_special_tokens=True))

Loss: 5.121203899383545
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


In [2]:
llm.config

Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "float16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 24,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "rms_n